# 07 — Supplementary PDF

Generates the supplementary PDF with RF maps and fitted parameters for all 31 verified m=1 simple cells. Output: `derived_data/m1_cells/supplementary_rf_gallery_m1_31.pdf`.

In [ ]:
from pathlib import Path
import sys, os

REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT / 'src'))
sys.path.insert(0, str(REPO_ROOT))

population_dir = REPO_ROOT / 'derived_data' / 'population'
gallery_dir    = REPO_ROOT / 'derived_data' / 'm1_cells'
DATASET_PATH   = gallery_dir / 'm1_neuron_dataset.pkl'
PDF_PATH       = gallery_dir / 'supplementary_rf_gallery_m1_31.pdf'

import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Ellipse

from rf_analysis.sparse_noise import _pixel_size

PDF_PATH     = gallery_dir / 'supplementary_rf_gallery_m1_31.pdf'

PIXEL_SIZE_DEG = _pixel_size('locally_sparse_noise_4deg')

plt.rcParams.update({
    'pdf.fonttype': 42, 'ps.fonttype': 42,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'savefig.dpi': 300, 'figure.dpi': 110,
})

with open(DATASET_PATH, 'rb') as f:
    dataset = pickle.load(f)

In [ ]:
csvs = sorted(population_dir.glob('rf_params_order_v2_container_*.csv'))
allp = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)

order_cols = ['cell_id', 'r2_m0', 'r2_m1', 'r2_m2', 'delta_r2_vs_m0',
              'derivative_order', 'cre_line', 'imaging_depth']
have = [c for c in order_cols if c in allp.columns]
orders = allp[have].drop_duplicates('cell_id').set_index('cell_id')

def order_stats(cell_id):
    """Return the stored per-order R^2 for a cell, or NaNs if absent."""
    if cell_id in orders.index:
        return orders.loc[cell_id]
    return pd.Series({c: np.nan for c in have if c != 'cell_id'})

In [ ]:
CMAP = 'RdBu_r'

def show_map(ax, M, title, vlim=None, cmap=CMAP):
    M = np.asarray(M, dtype=float)
    v = vlim if vlim is not None else np.abs(M).max()
    im = ax.imshow(M, cmap=cmap, vmin=-v, vmax=v, origin='upper',
                   interpolation='nearest')
    ax.set_title(title, fontsize=10, pad=5)
    ax.set_xticks([]); ax.set_yticks([])
    for s in ax.spines.values():
        s.set_linewidth(0.6); s.set_color('#444')
    return im

def fitted_axis_deg(rec):
    """Orientation of the model's differentiation axis, in degrees.

    sigma_phi_deg and sigma_orth_deg are the fitted sigma_u and sigma_v, which
    belong to theta, not to the lobe-geometry estimator phi.  fit_rf_by_order
    additionally swaps the two axes when sigma_v > sigma_u and reports theta as
    the MAJOR axis, so the differentiation axis is theta when
    sigma_phi >= sigma_orth and theta - 90 otherwise.
    """
    th = float(rec.get('theta_envelope', float('nan')))
    if not np.isfinite(th):
        return float('nan')
    su, sv = float(rec['sigma_phi_deg']), float(rec['sigma_orth_deg'])
    return th % 180.0 if su >= sv else (th - 90.0) % 180.0

def overlay_contours(ax, smooth, model):
    """Overlay the model's iso-response contours on the smoothed map, as in the
    review interface.  No ellipse: the lobe structure itself carries the shape.

    Contours are drawn at fixed fractions of the model's peak, ON lobe in warm
    lines and OFF lobe in cool lines, so the reader sees where the model places
    its mass rather than an abstracted outline.
    """
    M = np.asarray(model, dtype=float)
    peak = np.abs(M).max()
    if peak < 1e-12:
        return
    H, W = M.shape
    Y, X = np.mgrid[0:H, 0:W]
    levels = np.array([0.25, 0.5, 0.75]) * peak
    ax.contour(X, Y, M, levels=levels,
               colors='#8b1a1a', linewidths=0.9, zorder=4)
    ax.contour(X, Y, M, levels=-levels[::-1],
               colors='#12506b', linewidths=0.9, zorder=4)

def order_bar(ax, st):
    """Goodness of fit at each derivative order, with the selected one marked."""
    vals = [st.get('r2_m0', np.nan), st.get('r2_m1', np.nan), st.get('r2_m2', np.nan)]
    cols = ['#b8b8b8', '#1a7a4a', '#b8b8b8']
    ax.bar(['m=0', 'm=1', 'm=2'], vals, color=cols, width=0.62,
           edgecolor='#333', linewidth=0.5)
    ax.axhline(0.3, color='#c0392b', lw=0.8, ls='--')
    ax.text(-0.45, 0.315, 'QC', fontsize=7.5, color='#c0392b', ha='left')
    for i, v in enumerate(vals):
        if np.isfinite(v):
            ax.text(i, v + 0.015, f'{v:.3f}', ha='center', fontsize=8)
    finite = [v for v in vals if np.isfinite(v)]
    ax.set_ylim(0, max(0.42, max(finite) * 1.22) if finite else 0.42)
    ax.set_ylabel('$R^2$', fontsize=10)
    ax.set_title('Goodness of fit by derivative order', fontsize=10.5, pad=6)
    ax.tick_params(labelsize=9)
    for s in ('top', 'right'):
        ax.spines[s].set_visible(False)

def fmt(v, unit='', nd=2):
    """Format a value, or an em dash when it is missing."""
    try:
        f = float(v)
    except (TypeError, ValueError):
        return str(v) if v not in (None, '') else '\u2014'
    return '\u2014' if not np.isfinite(f) else f'{f:.{nd}f}{unit}'

In [ ]:
PAGE_W, PAGE_H = 11.69, 8.27
MAP_W_IN = 3.62
MAP_H_IN = MAP_W_IN * 9 / 16

def _ax(fig, x_in, y_in, w_in, h_in):
    """Add axes positioned in inches from the bottom-left of the page."""
    return fig.add_axes([x_in / PAGE_W, y_in / PAGE_H,
                         w_in / PAGE_W, h_in / PAGE_H])

def neuron_page(pdf, rec, idx):
    st = order_stats(int(rec['cell_id']))

    raw    = np.asarray(rec['rf_raw'], float)
    smooth = np.asarray(rec['rf_smooth'], float)
    model  = np.asarray(rec['model_map'], float)
    resid  = smooth - model
    vlim   = max(np.abs(smooth).max(), np.abs(model).max())

    fig = plt.figure(figsize=(PAGE_W, PAGE_H))

    L, GAP = 0.62, 0.34
    top_y  = PAGE_H - 1.05 - MAP_H_IN
    bot_y  = top_y - MAP_H_IN - 0.62

    fig.suptitle(rec['name'], fontsize=18, fontweight='bold',
                 x=L / PAGE_W, ha='left', y=0.965)
    fig.text(L / PAGE_W, 0.918,
             f"cell {int(rec['cell_id'])}   \u00b7   container {int(rec['container_id'])}"
             f"   \u00b7   verified first-order simple cell   \u00b7   {idx} of 31",
             fontsize=10, color='#555', ha='left', va='top')

    layout = [(0, 0, raw,    'Reconstructed map',             None),
              (1, 0, smooth, 'Smoothed ($\\sigma=0.75$ px)',  None),
              (0, 1, model,  'Fitted $m{=}1$ model',          vlim)]

    for col, row, M, title, vl in layout:
        x = L + col * (MAP_W_IN + GAP)
        y = top_y if row == 0 else bot_y
        ax = _ax(fig, x, y, MAP_W_IN, MAP_H_IN)
        show_map(ax, M, title, vl)

    axc = _ax(fig, L + 1 * (MAP_W_IN + GAP), bot_y, MAP_W_IN, MAP_H_IN)
    show_map(axc, smooth, 'Smoothed $+$ model contours', vlim * 2.2)
    overlay_contours(axc, smooth, model)

    bar_x = L + 2 * (MAP_W_IN + GAP) + 0.30
    bar_w = PAGE_W - 0.55 - bar_x
    ax = _ax(fig, bar_x, top_y - 0.30, bar_w, MAP_H_IN + 0.30)
    order_bar(ax, st)

    rows = [
        ('Geometric-mean scale  $\\sigma$',       fmt(rec['sigma_deg'], '\u00b0')),
        ('Along diff. axis  $\\sigma_\\varphi$',  fmt(rec['sigma_phi_deg'], '\u00b0')),
        ('Perpendicular  $\\sigma_\\perp$',       fmt(rec['sigma_orth_deg'], '\u00b0')),
        ('Elongation  $\\kappa_{\\mathrm{dir}}$', fmt(rec['kappa_dir'], nd=3)),
        ('Elongation  $\\kappa \\geq 1$',         fmt(rec['kappa'], nd=3)),
        ('Explained variance  $R^2$',            fmt(rec['r_squared'], nd=3)),
        ('Diff. direction  $\\varphi$',          fmt(rec['phi_deg'], '\u00b0', 1)),
        ('Envelope orientation  $\\theta$',
         fmt(rec.get('theta_envelope', float('nan')), '\u00b0', 1)),
        ('Centre  $(x_0, y_0)$',
         f"({fmt(rec['x0_deg'], chr(176), 1)}, {fmt(rec['y0_deg'], chr(176), 1)})"),
        ('Cre line',      str(st.get('cre_line', '\u2014'))),
        ('Imaging depth', fmt(st.get('imaging_depth', float('nan')), ' \u00b5m', 0)),
    ]
    axt = _ax(fig, L, 0.58, PAGE_W - 2 * L, bot_y - 0.58 - 0.30)
    axt.axis('off')
    axt.set_title('Fitted parameters', fontsize=11, loc='left', pad=6)
    for i, (k, v) in enumerate(rows):
        col, row = divmod(i, 6)
        x = 0.005 + col * 0.50
        y = 0.92 - row * 0.175
        axt.text(x, y, k, fontsize=9.5, va='center', color='#444')
        axt.text(x + 0.30, y, v, fontsize=9.5, va='center', fontweight='bold')

    fig.text(L / PAGE_W, 0.028,
             'Red, ON subfield; blue, OFF subfield; white, zero. '
             'The fitted map shares a colour scale with the smoothed map.\n'
             'Contours, model iso-response at 25, 50 and 75% of peak; '
             'warm lines the ON lobe, cool lines the OFF lobe.',
             fontsize=8, color='#666', ha='left')

    pdf.savefig(fig); plt.close(fig)

In [ ]:
def contents_page(pdf, dataset):
    tab = pd.DataFrame([{
        'Page': i + 2,
        'Name': r['name'],
        'Cell ID': int(r['cell_id']),
        'Container': int(r['container_id']),
        'sigma': r['sigma_deg'],
        'kappa_dir': r['kappa_dir'],
        'phi': r['phi_deg'],
        'R2': r['r_squared'],
    } for i, r in enumerate(dataset)])

    fig = plt.figure(figsize=(11.69, 8.27))
    fig.text(0.045, 0.955, 'Verified first-order simple cells',
             fontsize=19, fontweight='bold', ha='left', va='top')
    fig.text(0.045, 0.895,
             'Supplementary material: fitted receptive field models for the 31 '
             'neurons verified as first-order simple cells by manual review',
             fontsize=10.5, color='#444', ha='left', va='top')

    disp = tab.copy()
    disp['sigma']     = disp['sigma'].map('{:.2f}'.format)
    disp['kappa_dir'] = disp['kappa_dir'].map('{:.2f}'.format)
    disp['phi']       = disp['phi'].map('{:.1f}'.format)
    disp['R2']        = disp['R2'].map('{:.3f}'.format)
    disp.columns = ['p.', 'Name', 'Cell ID', 'Container',
                    'sigma', 'k_dir', 'phi', 'R2']

    half = int(np.ceil(len(disp) / 2))
    blocks = [(0.045, disp.iloc[:half]), (0.525, disp.iloc[half:])]
    for b, (x0, block) in enumerate(blocks):
        if block.empty:
            continue
        ax = fig.add_axes([x0, 0.13, 0.43, 0.70]); ax.axis('off')
        t = ax.table(cellText=block.values, colLabels=block.columns,
                     cellLoc='center', loc='upper center')
        t.auto_set_font_size(False); t.set_fontsize(7.0); t.scale(1, 1.42)
        for j, w in enumerate([0.07, 0.17, 0.22, 0.22, 0.12, 0.10, 0.10, 0.10]):
            for r in range(len(block) + 1):
                t[(r, j)].set_width(w)
        for (r, c), cell in t.get_celld().items():
            cell.set_linewidth(0.4); cell.set_edgecolor('#c8c8c8')
            if r == 0:
                cell.set_text_props(fontweight='bold'); cell.set_facecolor('#ececec')
            elif r % 2 == 0:
                cell.set_facecolor('#f8f8f8')

    fig.text(0.045, 0.085,
             f'Generated {date.today().isoformat()} from m1_neuron_dataset.pkl and the '
             f'order_full_v2 container parameter files.   '
             f'Analysis code: https://github.com/dmescherina/v1-rf-shapes-mice',
             fontsize=8, color='#666', va='top')

    pdf.savefig(fig); plt.close(fig)
    return tab

In [ ]:
order = sorted(dataset, key=lambda r: (int(r['container_id']), r['name']))

with PdfPages(PDF_PATH) as pdf:
    tab = contents_page(pdf, order)
    for i, rec in enumerate(order, start=1):
        neuron_page(pdf, rec, i)

    d = pdf.infodict()
    d['Title']    = 'Fitted receptive field models for 31 verified first-order simple cells'
    d['Author']   = 'Meshcherina, Auffarth, Lindeberg'
    d['Subject']  = 'Supplementary material'
    d['Keywords'] = ('mouse V1; simple cells; Gaussian derivative; '
                     'Allen Brain Observatory; receptive fields')

size_mb = PDF_PATH.stat().st_size / 1e6

tab.to_csv(gallery_dir / 'supplementary_rf_gallery_m1_31_table.csv', index=False)